# Notebook 3: Model Training

Loads the train/validation/test sets produced by `notebook_02_preprocessing` and trains models against them. This notebook does not repeat any cleaning or splitting logic, it only reads the already-processed CSVs, so it has no dependency on notebook 2's kernel state.

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

## 1. Load Preprocessed Data

Reads the three partitions notebook 2 saved to disk. `SaleYearMonth` comes back as a plain string after the CSV round-trip (not a `Period`) — harmless here since it's excluded from the model's features either way, kept only for traceability back to which month each row belongs to.

In [2]:
os.chdir(os.path.expanduser("~/Desktop/CAPropPredictor"))

housing_train_m1 = pd.read_csv("CRMLSCleaned/housingtrainm1.csv")
housing_val_m1 = pd.read_csv("CRMLSCleaned/housingvalm1.csv")
housing_test_m1 = pd.read_csv("CRMLSCleaned/housingtestm1.csv")

print(f"train: {housing_train_m1.shape}")
print(f"val:   {housing_val_m1.shape}")
print(f"test:  {housing_test_m1.shape}")

train: (128340, 25)
val:   (11890, 25)
test:  (11902, 25)


## 2. Linear Regression Baseline

A defensible baseline needs leakage-safe imputation and encoding, which is what `ColumnTransformer`/`Pipeline` are for — so this doubles as a first pass at Task 0's pipeline infrastructure, kept intentionally minimal since the goal here is a benchmark, not a tuned model.

`SaleYearMonth` is excluded from the feature list; it was only ever needed to make the chronological split possible, never as something the model should see as an input.

Categorical columns (`City`, `MLSAreaMajor`, `HighSchoolDistrict` in particular) are high-cardinality and go through plain one-hot encoding here as a known baseline simplification; CV-safe target encoding per the feature-engineering guidance is left as a future improvement.

R² on the test set is the metric the spec asks for; MAE, MAPE, and MdAPE are included alongside it since a single metric doesn't tell the whole story for skewed, multi-scale price data, and having them recorded now makes the eventual side-by-side comparison against advanced models possible without rerunning this cell later.

In [3]:
# SaleYearMonth: metadata carried through for the split, never a model input.
# ClosePrice: the target.
non_feature_columns = ["ClosePrice", "SaleYearMonth"]

numeric_feature_columns = [
    "Latitude", "Longitude",
    "ViewYN", "PoolPrivateYN", "AttachedGarageYN", "FireplaceYN", "NewConstructionYN",
    "ParkingTotal", "BathroomsTotalInteger", "BedroomsTotal", "MainLevelBedrooms", "GarageSpaces",
    "LivingArea", "LotSizeSquareFeet", "AssociationFee", "YearBuilt",
    "Levels", "Stories",
]
categorical_feature_columns = ["City", "CountyOrParish", "MLSAreaMajor", "HighSchoolDistrict", "Flooring"]
feature_columns = numeric_feature_columns + categorical_feature_columns

# catches a typo or a column that's neither a listed feature nor listed metadata
assert set(feature_columns) == set(housing_train_m1.columns) - set(non_feature_columns), (
    "feature_columns doesn't match housing_train_m1's actual columns -- check for "
    "a typo, or a column that needs to be added to one list or the other."
)

preprocessor = ColumnTransformer(transformers=[
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_feature_columns),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("encode", OneHotEncoder(handle_unknown="ignore")),
    ]), categorical_feature_columns),
])

baseline_pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", LinearRegression()),
])

X_train, y_train = housing_train_m1[feature_columns], housing_train_m1["ClosePrice"]
X_val, y_val = housing_val_m1[feature_columns], housing_val_m1["ClosePrice"]
X_test, y_test = housing_test_m1[feature_columns], housing_test_m1["ClosePrice"]

# fit on train only -- val/test are transformed using statistics learned from
# train alone, since preprocess is inside the same Pipeline that gets fit here
baseline_pipeline.fit(X_train, y_train)

def evaluate(pipeline, X, y, label):
    preds = pipeline.predict(X)
    r2 = r2_score(y, preds)
    mae = mean_absolute_error(y, preds)
    mape = mean_absolute_percentage_error(y, preds)
    mdape = np.median(np.abs((y - preds) / y))
    print(f"{label:>5s}: R2={r2:.4f}  MAE=${mae:,.0f}  MAPE={mape:.2%}  MdAPE={mdape:.2%}")
    return {"r2": r2, "mae": mae, "mape": mape, "mdape": mdape}

print("Baseline Linear Regression results:")
train_metrics = evaluate(baseline_pipeline, X_train, y_train, "train")
val_metrics = evaluate(baseline_pipeline, X_val, y_val, "val")
test_metrics = evaluate(baseline_pipeline, X_test, y_test, "test")

Baseline Linear Regression results:
train: R2=0.8266  MAE=$232,591  MAPE=22.25%  MdAPE=15.90%
  val: R2=0.8223  MAE=$247,420  MAPE=22.98%  MdAPE=16.19%
 test: R2=0.8217  MAE=$248,884  MAPE=22.68%  MdAPE=16.10%


In [4]:
baseline_results = {
    "model": "LinearRegression",
    "train": train_metrics,
    "val": val_metrics,
    "test": test_metrics,
}
baseline_results

{'model': 'LinearRegression',
 'train': {'r2': 0.8266306771364224,
  'mae': 232591.12288560916,
  'mape': 0.22251765483661193,
  'mdape': np.float64(0.15899044860562425)},
 'val': {'r2': 0.8222956893138691,
  'mae': 247420.077640253,
  'mape': 0.229771633968163,
  'mdape': np.float64(0.16188053132415386)},
 'test': {'r2': 0.8216869592348737,
  'mae': 248883.89974544835,
  'mape': 0.22676005127073165,
  'mdape': np.float64(0.16103182364593047)}}

## Next Steps: Pipeline Edits and Feature Enhancements

Week 5 will start by comparing the Linear Regression baseline against at least one tree-based model (Random Forest or a gradient-boosted variant). Which one wins, and by how much, directly shapes where feature engineering effort goes next, not just whether more features get added. If a tree-based model beats Linear Regression by a wide margin, that gap is itself informative: it means there's non-linear structure or feature interactions in the data that Linear Regression can't represent on its own, which would justify spending effort on explicit transforms (log-price, interaction terms, binning) rather than just adding raw new columns. If the gap is small, that suggests the linear model is already capturing most of the available signal, and effort is better spent introducing genuinely new information (like the location and temporal features below) than reshaping what's already there. Either way, the model comparison is a diagnostic, not just a leaderboard.

**1a. Geocoding to fix incorrect coordinates, not just fill missing ones.** The Hemet placeholder cluster found earlier is the clearest example: 102 listings in Riverside County sharing one exact coordinate look complete in the data (no NaN to catch) but are silently wrong. The geocoding pipeline already built should be used to detect and correct this class of error broadly, not only to fill rows where Latitude/Longitude are outright missing.

**1b. Confirmed coordinates unlock a school district join.** Once geocoding is reliable, Latitude/Longitude can be joined against a school district shapefile to derive a clean, authoritative district feature. This is a strong candidate for a high-quality feature and a likely replacement for the raw `HighSchoolDistrict`/`ElementarySchoolDistrict` text columns, which had null rates high enough to get dropped outright in preprocessing.

**2. A temporal feature for the sale's closing month.** This is not leakage: which month a deal closes is often a real preference known at negotiation time (tax timing, a lease ending, a school year starting), not something only knowable after the sale has already happened. That's a different category from the fields already excluded for leakage (`DaysOnMarket`, closing/contract dates, price-reduction flags), which only exist because a sale process unfolded and reflect its outcome. Closing month can be treated more like an input condition a buyer or seller sets going in, and the seasonality it captures (spring/summer price premiums, slower winter markets) is a real market effect, not an artifact of any individual transaction. Sine/cosine cyclical encoding avoids an artificial ordinal jump from December to January.

**3. Other likely-needed improvements:**
- Investigate the absurd bathroom/bedroom values flagged earlier (counts upward of 100) and decide on a cap or a hard exclusion rule.
- Move from flat median imputation to group-wise imputation (e.g. by `PropertySubType` or neighborhood) for better fidelity, per the missingness-handling guidance.
- Add missing-indicator flags where a null itself is informative, e.g. a missing `AssociationFee` most likely means no HOA rather than an unknown fee.
- Revisit `City`/`MLSAreaMajor`/`HighSchoolDistrict` moving from one-hot to CV-safe target encoding, especially once the shapefile-derived district feature (1b) may replace the messiest of these anyway.